<h1><center>Harness Engineering</center></h1>

In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import pandas as pd
from claude_agent_sdk import AgentDefinition, AssistantMessage, ClaudeAgentOptions, ResultMessage, TextBlock, query
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)

if os.getenv('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY is set')
else:
    print('ANTHROPIC_API_KEY is not set')

ANTHROPIC_API_KEY is set


## Part 1: The Claims Desk

In [3]:
DATA = Path("../../../data")

def load(name):
    """ Read one of the JSON seed files. """
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def get_sample_data(tiny=False, held_out=False):
    """ Return a list of claims and the handlers'n decision on them keyed by claim id. 
    
    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """

    claims = load('evals/held_out_claims.json' if held_out else 'claims.json')
    decisions = load('evals/held_out_decisions.json' if held_out else 'evals/decisions.json')
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels
   

In [4]:
claims, labels = get_sample_data(tiny=True)

for claim in claims:
    print(claim['claim_id'], claim['summary'])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


In [5]:
DESK = Path('lab1_desk')

def make_desk(claims):
    """ Build a fresh claims desk with the claims to decide and everything needed to decide them. """
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ['claims', 'policies', 'customers', 'decisions']:
        (DESK / folder).mkdir(parents=True)
    for claim in claims: 
        (DESK / 'claims' / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2),
         encoding='utf-8')
    for policy in load('policies.json'):
        (DESK /'policies' / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), 
        encoding='utf-8')
    for customer in load('customers.json'):
        (DESK / 'customers' / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2),
        encoding='utf-8')
    for name in ['policy_wording.md', 'claims_manual.md', 'company.json']:
        shutil.copy(DATA / name, DESK / name)

make_desk(claims)
sorted(path.name for path in DESK.iterdir())



['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

In [6]:
INSTRUCTIONS = """ 
You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.
Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue.
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids 
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a task list with one task for each claim, and gie each claim to the claim-handler agent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """
You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.
Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.
Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""


In [8]:
TOOLS =['Read', 
        'Glob', 
        'Grep', 
        'Write',
        'Task',
        'TaskCreate',
        'TaskUpdate',
        'TaskList',
        'TaskGet']

claim_handler= AgentDefinition(
    description='Decides one claim and writes its decision file. Give it the claim id.',
    prompt=HANDLER_INSTRUCTIONS,
    tools=['Read',
            'Glob',
            'Grep',
            'Write'])

options = ClaudeAgentOptions(
    model='claude-sonnet-5-5',
    effort='medium',
    cwd=str(DESK),
    system_prompt={
        'type': 'preset',
        'preset': 'claude_code',
        'append': INSTRUCTIONS
    },
    tools=TOOLS,
    allowed_tools=TOOLS,
    agents={'claim-handler': claim_handler},
    permission_mode='dontAsk',
    setting_sources=[],
    env={'CLAUDE_CODE_DISABLE_AUTO_MEMORY': '1', 'ENABLE_CLAUDEAI_MCP_SERVERS':'false'},
    max_budget_usd=1.00,
)

In [9]:
import sys

if sys.platform == 'win32':
    import asyncio
    import threading

    import claude_agent_sdk

    async def query(*, prompt, options):
        """ The SDK's query, run an event loop of its own in a thread, passing each message back here. """
        loop = asyncio.get_running_loop()
        messages = asyncio.Queue()

        async def run():
            try:
                async for message in claude_agent_sdk.query(prompt=prompt, options=options):
                    loop.call_soon_threadsafe(messages.put_nowait, message)
            except Exception as error:
                loop.call_soon_threadsafe(messages.put_nowait, None)
        threading.Thread(target=asyncio.run, 
                         args=(run(),), kwargs={'loop_factory': asyncio.ProactorEventLoop}).start()

        while (message := await messages.get()) is not None:
            if isinstance(message, Exception):
                raise message
            yield message

    print('Applied the Windows adjustment')
else:
    print('Not Windows, so nothing to do here.')


Not Windows, so nothing to do here.


## Part 3: Working The Queue

In [10]:
def describe(tool_call):
    """ A short line about one tool call."""
    args = tool_call.input
    if'file_path' in args:
        return f"{tool_call.name} {Path(args['file_path']).name}"
    if tool_call.name in ('Agent', 'Task'):
        return f"{tool_call.name}: {args.get('description')}"
    if tool_call.name == 'TaskCreate':
        return f"TaskCreate: {args.get('subject')}"
    if tool_call.name == 'TaskUpdate':
        return f"TaskUpdate: task {args.get('taskId')} {args.get('status', '')}"
    if 'pattern' in args:
        return f"{tool_call.name} {args['pattern']}"
    return tool_call.name

In [15]:
result = None
start = time.time()

async for message in query(prompt='Please work the queue.', options=options):
    if isinstance(message, AssistantMessage):
        indent = "      " if message.parent_tool.use_id else ""
        for block in message.content:
            if isinstance(block, TextBlock):
                print(indent + describe(block))
            elif isinstance(block, TextBlock) and block.text.strip():
                print(indent + block.text.strip())
    elif isinstance(message, ResultMessage):
        result = message

print(f"\nFinished in {time.time() - start:.0f} seconds for ${result.total_cost_usd:.2f}")


AttributeError: 'AssistantMessage' object has no attribute 'parent_tool'

## Part 4: How it did

In [14]:
def is_right(answer, label):
    """ True when the decision matches the handler's and the payout is within a dollar. """
    return answer.get('decision') == label['decision'] and abs(answer.get('payout', 0) - label['payout']) <= 1

rows = []
for claim_id, label in labels.items():
    path = DESK / 'decisions' / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'decision': 'missing', 'payout': 0}
    rows.append({
        'claim': claim_id,
        'decision': answer.get('decision'),
        'payout': answer.get('payout'),
        "handler's decision": label['decision'],
        "handler's payout": label['payout'],
        'right': is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

0 of 5 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,missing,0,approve,50.0,False
1,CLM-1015,missing,0,decline,0.0,False
2,CLM-1017,missing,0,refer,0.0,False
3,CLM-1020,missing,0,approve,240.0,False
4,CLM-1021,missing,0,approve,825.0,False
